# 네이버 뉴스 크롤링 - Graph DB 데이터 수집

네이버 뉴스에서 분야별 헤드라인 기사 수집

- **정치**: https://news.naver.com/section/100
- **경제**: https://news.naver.com/section/101
- **사회**: https://news.naver.com/section/102
- **생활/문화**: https://news.naver.com/section/103
- **IT/과학**: https://news.naver.com/section/105
- **세계**: https://news.naver.com/section/104

## 수집 항목 (Articles.xlsx)
- article_id: 기사 고유 ID
- title: 기사 제목
- content: 기사 본문
- url: 기사 URL
- published_date: 발행일
- source: 언론사
- author: 기자명
- category: 카테고리

## 1. 필요한 패키지 설치 및 임포트

In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import pandas as pd
import time
from datetime import datetime
import re

## 2. 카테고리 설정 및 Selenium 초기화

In [4]:
categories = {
    '정치': 'https://news.naver.com/section/100',
    '경제': 'https://news.naver.com/section/101',
    '사회': 'https://news.naver.com/section/102',
    '생활/문화': 'https://news.naver.com/section/103',
    'IT/과학': 'https://news.naver.com/section/105',
    '세계': 'https://news.naver.com/section/104'
}

NUM_ARTICLES_PER_CATEGORY = 10

In [5]:
service = Service(ChromeDriverManager().install())
options = webdriver.ChromeOptions()
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(service=service, options=options)

## 3. 기사 수집 함수 정의

In [6]:
def get_article_links(driver, category_url, num_articles):
    driver.get(category_url)
    time.sleep(3)

    article_links = []

    try:
        selectors = [
            'a.sa_text_lede',
            'a.sa_text_strong',
            '.sa_text a',
            '.cluster_text_headline a',
            '.cluster_text_lede a'
        ]

        for selector in selectors:
            elements = driver.find_elements(By.CSS_SELECTOR, selector)
            for element in elements:
                url = element.get_attribute('href')
                if (url and 'news.naver.com' in url and '/article/' in url
                    and '/comment/' not in url  # 댓글 페이지만 제외
                    and url not in article_links):
                    article_links.append(url)
                    if len(article_links) >= num_articles:
                        break
            if len(article_links) >= num_articles:
                break

        print(f"✓ {len(article_links)}개의 기사 링크 수집 완료")

    except Exception as e:
        print(f"✗ 기사 링크 수집 실패: {e}")

    return article_links[:num_articles]

In [7]:
def parse_article_detail(driver, article_url, category):
    driver.get(article_url)
    time.sleep(1.5)

    article_data = {
        'article_id': '',
        'title': '',
        'content': '',
        'url': article_url,
        'published_date': '',
        'source': '',
        'author': '',
        'category': category
    }

    try:
        # 기사 ID 생성 (URL에서 추출)
        match = re.search(r'article/(\d+)/(\d+)', article_url)
        if match:
            article_data['article_id'] = f"ART_{match.group(1)}_{match.group(2)}"
        else:
            article_data['article_id'] = f"ART_{datetime.now().strftime('%Y%m%d%H%M%S')}"

        # 제목
        title_selectors = [
            '#title_area span',
            '#ct .media_end_head_headline',
            '.media_end_head_headline',
            'h2#title_area',
            '.news_end_title'
        ]

        for selector in title_selectors:
            try:
                title_element = driver.find_element(By.CSS_SELECTOR, selector)
                if title_element.text.strip():
                    article_data['title'] = title_element.text.strip()
                    break
            except:
                continue

        # 본문
        content_selectors = [
            '#dic_area',
            'article#dic_area',
            '.go_trans._article_content',
            '._article_body_contents'
        ]

        for selector in content_selectors:
            try:
                content_element = driver.find_element(By.CSS_SELECTOR, selector)
                if content_element.text.strip():
                    article_data['content'] = content_element.text.strip()
                    break
            except:
                continue

        # 언론사
        try:
            source_element = driver.find_element(By.CSS_SELECTOR, 'a.media_end_head_top_logo img')
            article_data['source'] = source_element.get_attribute('alt')
        except:
            try:
                source_element = driver.find_element(By.CSS_SELECTOR, '.media_end_head_top_logo_text')
                article_data['source'] = source_element.text.strip()
            except:
                pass

        # 발행일
        try:
            date_element = driver.find_element(By.CSS_SELECTOR, 'span.media_end_head_info_datestamp_time, span[data-date-time]')
            date_text = date_element.get_attribute('data-date-time') or date_element.text
            article_data['published_date'] = date_text.strip()
        except:
            article_data['published_date'] = datetime.now().strftime('%Y-%m-%d %H:%M')

        # 기자명
        try:
            author_element = driver.find_element(By.CSS_SELECTOR, 'em.media_end_head_journalist_name, span.byline_s')
            article_data['author'] = author_element.text.strip()
        except:
            pass

    except Exception as e:
        print(f"  ✗ 파싱 오류: {e}")

    return article_data

## 4. 전체 카테고리 크롤링 실행

In [8]:
# 전체 기사 데이터를 저장할 리스트
all_articles = []

# 카테고리별 크롤링
for category_name, category_url in categories.items():
    print(f"\n{'='*60}")
    print(f"[{category_name}] 카테고리 수집 시작...")
    print(f"{'='*60}")

    # 1단계: 기사 링크 수집
    article_links = get_article_links(driver, category_url, NUM_ARTICLES_PER_CATEGORY)

    # 2단계: 각 기사 상세 정보 수집
    for idx, article_url in enumerate(article_links, 1):
        print(f"  [{idx}/{len(article_links)}] {article_url}")
        article_data = parse_article_detail(driver, article_url, category_name)

        if article_data['title']:  # 제목이 있는 경우만 추가
            all_articles.append(article_data)
            print(f"  ✓ 수집 완료: {article_data['title'][:50]}...")
        else:
            print(f"  ✗ 수집 실패 - 제목을 찾을 수 없습니다.")

        time.sleep(0.5)


[정치] 카테고리 수집 시작...
✓ 10개의 기사 링크 수집 완료
  [1/10] https://n.news.naver.com/mnews/article/009/0005645981
  ✓ 수집 완료: ‘징계 효력정지’ 법원 결정에...배현진 “장동혁 지도부 반성하라” 일갈...
  [2/10] https://n.news.naver.com/mnews/article/119/0003065689
  ✓ 수집 완료: 李대통령, 국무회의서 '사법 3법' 의결…거부권 안썼다...
  [3/10] https://n.news.naver.com/mnews/article/657/0000048934
  ✓ 수집 완료: 중동 상황 악화로 여행·항공·숙박 '소비자 피해주의보' 발령...
  [4/10] https://n.news.naver.com/mnews/article/214/0001484268
  ✓ 수집 완료: 주한미군 미사일 중동으로?‥'패트리엇'·'다연장포' 유력...
  [5/10] https://n.news.naver.com/mnews/article/011/0004596220
  ✓ 수집 완료: 청와대 복귀에도 북악산 24시간 개방 유지…6개 권역 안내소 다시 운영도...
  [6/10] https://n.news.naver.com/mnews/article/022/0004110444
  ✓ 수집 완료: [속보] 李대통령 “빈말하지 않는다…규칙 어겨 이익 보는 시대는 갔다”...
  [7/10] https://n.news.naver.com/mnews/article/008/0005325883
  ✓ 수집 완료: 송언석 "'사법파괴 3대 악법 철폐' '공소 취소 저지' 투쟁 지속 전개"...
  [8/10] https://n.news.naver.com/mnews/article/437/0000481494
  ✓ 수집 완료: 당론 채택에도 법사위 강경파 "검사들 자르고 다시 뽑아야"...
  [9/10] https://n.news.naver.com/mnews/article/449

## 5. 수집 결과 확인

In [9]:
df_articles = pd.DataFrame(all_articles)

In [10]:
df_articles

,article_id,title,content,url,published_date,source,author,category
0,ART_009_0005645981,‘징계 효력정지’ 법원 결정에...배현진 “장동혁 지도부 반성하라” 일갈,[연합뉴스]\n국민의힘 배현진 의원은 자신에게 내려진 당원권 정지 처분의 효력을 정...,https://n.news.naver.com/mnews/article/009/000...,2026-03-05 18:50:11,매일경제,방영덕 기자,정치
1,ART_119_0003065689,"李대통령, 국무회의서 '사법 3법' 의결…거부권 안썼다",법왜곡죄·재판소원·대법관 증원법 통과\n이재명 대통령이 5일 청와대에서 열린 임시 ...,https://n.news.naver.com/mnews/article/119/000...,2026-03-05 13:58:09,데일리안,김은지 기자,정치
2,ART_657_0000048934,중동 상황 악화로 여행·항공·숙박 '소비자 피해주의보' 발령,자료화면\n\n\n한국소비자원과 공정거래위원회는 최근 무력 충돌로 중동 지역 상황이...,https://n.news.naver.com/mnews/article/657/000...,2026-03-05 19:09:11,대구MBC,도건협 기자,정치
3,ART_214_0001484268,주한미군 미사일 중동으로?‥'패트리엇'·'다연장포' 유력,"주한미군 미사일 중동으로?‥'패트리엇'·'다연장포' 유력\nMBC뉴스\n재생\n6,...",https://n.news.naver.com/mnews/article/214/000...,2026-03-05 22:17:47,MBC,변윤재 기자,정치
4,ART_011_0004596220,청와대 복귀에도 북악산 24시간 개방 유지…6개 권역 안내소 다시 운영도,대통령경호처 “일상 존중”…추가 탐방로 신설\n국가유산청은 한양도성 안내소 6곳 운...,https://n.news.naver.com/mnews/article/011/000...,2026-03-05 18:01:12,서울경제,최수문 기자,정치
5,ART_022_0004110444,[속보] 李대통령 “빈말하지 않는다…규칙 어겨 이익 보는 시대는 갔다”,이재명 대통령. 뉴시스\n이재명 대통령은 5일 “부당한 시스템에 의존하고 정당한 정...,https://n.news.naver.com/mnews/article/022/000...,2026-03-05 15:12:10,세계일보,박윤희 기자,정치
6,ART_008_0005325883,"송언석 ""'사법파괴 3대 악법 철폐' '공소 취소 저지' 투쟁 지속 전개""",[the300]\n(서울=뉴스1) 신웅수 기자 = 송언석 국민의힘 원내대표가 5일 ...,https://n.news.naver.com/mnews/article/008/000...,2026-03-05 14:59:32,머니투데이,정경훈 기자,정치
7,ART_437_0000481494,"당론 채택에도 법사위 강경파 ""검사들 자르고 다시 뽑아야""",국회 법제사법위원회 김용민 법안심사 제1 소위위원장이 지난 2월 20일 국회에서 소...,https://n.news.naver.com/mnews/article/437/000...,2026-03-05 10:51:34,JTBC,유혜은 기자,정치
8,ART_449_0000337236,"한국 유조선 7척, 호르무즈 해협에 갇혔다","與 김영배 ""현재 총 7척이 호르무즈 해협에 있다""\n호르무즈 해협 봉쇄…국내 반도...",https://n.news.naver.com/mnews/article/449/000...,2026-03-05 19:02:49,채널A,,정치
9,ART_001_0015939319,"이준석 ""부정선거 음모론 국힘…보수탈 쓴 체제 허무는 급진세력""",개혁신당 최고위원회의\n(서울=연합뉴스) 신현우 기자 = 개혁신당 이준석 대표가 5...,https://n.news.naver.com/mnews/article/001/001...,2026-03-05 10:53:28,연합뉴스,박수윤 기자,정치


## 6. Excel 파일로 저장

In [11]:
output_filename = f"Articles_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
df_articles.to_excel(output_filename, index=False, engine='openpyxl')

In [12]:
driver.quit()